# Ruse

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import time
import shutil
import pathlib
import tempfile
from pprint import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import matplot2tikz 

from helpers import *

## Default experiment configuration

In [ ]:
default_config = {
    "timeout": datetime.timedelta(hours=1),
    "max_iterations": 6,
    "max_sequence_size": 3,
    "max_mutations": 3,
    "max_memory_usage": "100GiB",
    "workers_count": 64
}

## Helper functions

#### Some notebook helper functions

In [ ]:
def display_df(df: pd.DataFrame):
    s = df.style.set_properties(**{'text-align': 'left'})
    s.set_table_styles([dict(selector='th', props=[('text-align', 'left')])])
    html_table = s.to_html(justify='left', notebook=True, show_dimensions=True, index=False)

    scrollable_html = f"""
<div style="max-height: 300px; overflow: auto;">
    {html_table}
</div>
"""
    display(HTML(scrollable_html))

class StopExecution(Exception):
    def _render_traceback_(self):
        return []

#### Parse results

In [ ]:
def sort_by_oop_and_side_effects_and_name(df: pd.DataFrame, inplace=False):
    oop_category_order = {
        "Full OOP": 0,
        "Primitive Objects": 1,
        "Primitive": 2,
    }

    side_effects_order = {
        "With Side Effects": 0,
        "Possibly With Side Effects": 1,
        "Without Side Effects": 2
    }

    def sort_func(col):
        if col.name == "oop_category":
            return col.apply(lambda x: oop_category_order[x])
        if col.name == "side_effects":
            return col.apply(lambda x: side_effects_order[x])
        if col.name == "name":
            return col

    return df.sort_values(by=["oop_category", "side_effects", "name"], key=sort_func, inplace=inplace)

def sort_by_found_time(df: pd.DataFrame):
    def sort_func(col):
        if col.name == "found":
            return col.apply(lambda x: 0 if pd.notna(x) else 1)
        if col.name == "total_time":
            return col

    df.sort_values(by=["found", "total_time"], key=sort_func, inplace=True)

def get_sy_info(path):
    with open(path, "r") as f:
        return json.load(f)

def create_tasks_dataframe(tasks, dry_run) -> pd.DataFrame:
    df = pd.DataFrame(tasks)
    df["name"] = df["path"].apply(lambda x: os.path.basename(x).split(".")[0])
    df["total_time"] = df["total_time"].apply(lambda x: pd.Timedelta(seconds=x["secs"], nanoseconds=x["nanos"]).total_seconds() if pd.notnull(x) else x).astype(dtype="float64")
    df["path"] = df["path"].apply(lambda x: os.path.relpath(os.path.abspath(x), get_workspace_root()))
    df.rename(columns={"total_statistics": "stats"}, inplace=True)

    if "category" in df.columns:
        df["oop_category"] = df["category"].apply(lambda x: x.split(":")[0])
        df["side_effects"] = df["category"].apply(lambda x: x.split(":")[1])
        df.pop("category")

    if not dry_run:
        df = pd.json_normalize(df.to_dict(orient="records"))
        df["found.num_mutations"] = df["found.num_mutations"].astype(dtype="Int64")
        if "found" in df.columns:
            df.pop("found") 
        df.rename(columns={"found.found": "found"}, inplace=True)
        sort_by_found_time(df)

    ordered_columns = ['name',
                       'total_time',
                       'found',
                       'error',
                       'found.has_side_effects',
                       'found.num_mutations',
                       'path',
                       'source',
                       'category',
                       'oop_category',
                       'side_effects',
                       'opcode_count',
                       'max_vm_usage',
                       'stats.Evaluated',
                       'stats.BankSize',
                       'stats.FoundContextCount',
                       'stats.MaxMutatingOpcodes',
                       'stats.MaxDepth',
                       'stats.MaxSize',
                       'string_literals',
                       'num_literals',
                       'iterations']

    ordered_columns = [col for col in ordered_columns if col in df.columns]
    df = df[ordered_columns]

    return df

def parse_results(results_dir, dry_run):
    tasks = []
    metadata = None
    for file in os.listdir(results_dir):
        if file == 'metadata.json':
            with open(os.path.join(results_dir, file), "r") as f:
                metadata = json.load(f)
        elif file.startswith('task_') and file.endswith('.json'):
            with open(os.path.join(results_dir, file), "r") as f:
                tasks.append(json.load(f))
    df = create_tasks_dataframe(tasks, dry_run)

    df["expected"] = df["path"].apply(lambda x: get_sy_info(os.path.join(get_workspace_root(), x))["solution"]["expected"])
    df["examples_count"] = df["path"].apply(lambda x: len(get_sy_info(os.path.join(get_workspace_root(), x))["examples"]))
    df["var_count"] = df["path"].apply(lambda x: len(get_sy_info(os.path.join(get_workspace_root(), x))["variables"]))

    metadata["tasks_count"] = len(df)
    metadata["total_time"] = datetime.timedelta(seconds=df["total_time"].sum())
    metadata["error_count"] = df["error"].notna().sum()
    return metadata, df

In [ ]:
class TaskParserError(Exception):
    def __init__(self, errors):
        self.errors = errors
        
    def _render_traceback_(self):
        return []


def get_all_tasks(tasks):
    with tempfile.TemporaryDirectory() as temp_dir:
        results_dir = os.path.join(temp_dir, "results")

        run_ruse(tasks, results_dir, dry_run=True, in_background=False)
        metadata, tasks = parse_results(results_dir, dry_run=True)

    if metadata["error_count"] > 0:
        print("Errors:")
        raise TaskParserError(tasks[tasks["error"].notna()][["name", "error"]])

    return sort_by_oop_and_side_effects_and_name(tasks)

In [ ]:
def save_tasks_latex_table(df: pd.DataFrame, filename: str, caption: str, label: str):

    columns = ["name", "side_effects", "examples_count", "var_count"]
    headers = ["Task", "Side Effects", "\#Examples", "\#Variables"]
    column_types = ["l", "c", "c", "c"]
    df = df[columns].copy()

    def side_effects_formatter(x):
        if x == "Without Side Effects":
            return "No"
        elif x == "With Side Effects":
            return "Yes"
        elif x == "Possibly With Side Effects":
            return "Possibly"
        raise ValueError(f"Unknown side effect: {x}")

    df["side_effects"] = df["side_effects"].apply(side_effects_formatter)
    df["name"] = df["name"].apply(lambda x: x.replace("_", "\\_"))

    latex = ""
    latex += f"\\begin{{longtable}}{{| l | c c c |}}\n"
    latex += f"\t\\caption{{{caption}. \label{{{label}}} }} \\\\\n\n"
    
    latex += '\t\\hline\n'
    latex += f'\t\\multicolumn{{{len(df.columns)}}}{{| c |}}{{Begin of Table}} \\\\\n'
    latex += '\t\\hline\n'
    latex += "\t" + " & ".join([f"\\textbf{{{h}}}" for h in headers])
    latex += " \\\\\n"
    latex += '\t\\hline\n'
    latex += '\t\\endfirsthead\n\n'

    latex += '\t\\hline\n'
    latex += f'\t\\multicolumn{{{len(df.columns)}}}{{| c |}}{{Continuation of Table \\ref{{{label}}}}} \\\\\n'
    latex += '\t\\hline\n'
    latex += "\t" + " & ".join([f"\\textbf{{{h}}}" for h in headers])
    latex += " \\\\\n"
    latex += '\t\\hline\n'
    latex += '\t\\endhead\n\n'

    latex += '\t\\hline\n'
    latex += '\t\\endfoot\n'
    latex += '\t\\hline\\hline\n'
    latex += '\t\\endlastfoot\n\n'

    for _, row in df.iterrows():
        latex += "\t" + " & ".join([f"{cell}" for cell in row])
        latex += " \\\\\n"
    latex += f"\\end{{longtable}}\n"
    with open(filename, "w") as f:
        f.write(latex)

def save_figure_as_latex(filename: str, caption: str = None, label: str = None, **kwargs):
    tikz_plot = matplot2tikz.get_tikz_code(filepath=filename, **kwargs)
    latex = "\\begin{figure}[H]"
    latex += tikz_plot
    if caption:
        latex += f"\\caption{{{caption}}}\n"
    if label:
        latex += f"\\label{{{label}}}\n"
    latex += "\\end{figure}\n"
    with open(filename, "w") as f:
        f.write(latex)

## Experiments

### Benchmarks

We divide the benchmarks into two categories:
* Tasks that use only primitive and built-in objects (Like Array and Set)
* Tasks that use custom User classes

In [ ]:
tasks_paths = [
    os.path.join(get_workspace_root(), "tasks/benchmarks/"),
]

try:
    tasks = get_all_tasks(tasks_paths)
except Exception as e:
    if isinstance(e, TaskParserError):
        print(e)
        raise StopExecution
    else: 
        raise e

full_oop_tasks = tasks[tasks["oop_category"] == "Full OOP"]
primitive_tasks = tasks[tasks["oop_category"] != "Full OOP"]

#### Primitive tasks
We compare the primitive tasks against <span style="font-variant:small-caps;">SoBEq</span> and <span style="font-variant:small-caps;">FrAngel</span>

In [ ]:
primitive_table = primitive_tasks.copy()
primitive_table.drop(primitive_table[primitive_table["name"] == "getElementById"].index, inplace=True)

display_df(primitive_table[["name", "side_effects", "expected"]])
save_tasks_latex_table(primitive_table, "results/figures/primitive_tasks.tex", caption="Primitive and built-in objects tasks", label="primitive_tasks")

pure_primitive_solutions = primitive_table[primitive_table["side_effects"] == "Without Side Effects"]

print(f"{len(primitive_table)} primitive tasks")
print(f"{len(pure_primitive_solutions)} tasks expected solution is pure")
print(f"{len(primitive_table) - len(pure_primitive_solutions)} tasks expected solution can be solved with side effects")

#### Full OOP tasks

We compare the full oop tasks against <span style="font-variant:small-caps;">FrAngel</span>

In [ ]:
full_oop_table = full_oop_tasks.copy()
full_oop_table.drop(full_oop_table[full_oop_table["name"] == "sypet_10_scale"].index, inplace=True)
full_oop_table.drop(full_oop_table[full_oop_table["name"] == "inc_tree_values_min_ops"].index, inplace=True)
full_oop_table.drop(full_oop_table[full_oop_table["name"] == "inc_tree_values_connected_min_ops"].index, inplace=True)

pure_oop_solutions = full_oop_table[full_oop_table["side_effects"] == "Without Side Effects"]

display_df(full_oop_table[["name", "side_effects", "expected"]])
save_tasks_latex_table(full_oop_table, "results/figures/full_oop_tasks.tex", caption="Full OOP tasks", label="full_oop_tasks")

print(f"{len(full_oop_table)} full oop tasks")
print(f"{len(pure_oop_solutions)} tasks expected solution is pure")
print(f"{len(full_oop_table) - len(pure_oop_solutions)} tasks expected solution can be solved with side effects")


### RQ1 - How does <span style="font-variant:small-caps;">Ruse</span> compare to <span style="font-variant:small-caps;">FrAngel</span> and <span style="font-variant:small-caps;">SoBEq</span>?

In [ ]:
metadata, ruse_tasks = parse_results("results/ruse_all_tasks_results", dry_run=False)
ruse_tasks.to_csv("results/ruse_all_tasks_results.csv", index=False)

ruse_tasks.drop(ruse_tasks[ruse_tasks["name"] == "getElementById"].index, inplace=True)
ruse_tasks.drop(ruse_tasks[ruse_tasks["name"] == "sypet_10_scale"].index, inplace=True)

ruse_full_oop_tasks = ruse_tasks[ruse_tasks["oop_category"] == "Full OOP"].copy()
ruse_primitive_tasks = ruse_tasks[ruse_tasks["oop_category"] != "Full OOP"].copy()

ruse_full_oop_tasks.drop(ruse_full_oop_tasks[ruse_full_oop_tasks["name"] == "inc_tree_values_min_ops"].index, inplace=True)
ruse_full_oop_tasks.drop(ruse_full_oop_tasks[ruse_full_oop_tasks["name"] == "inc_tree_values_connected_min_ops"].index, inplace=True)

ruse_full_oop_solved_tasks = ruse_full_oop_tasks[ruse_full_oop_tasks["found"].notnull()]
ruse_primitive_solved_tasks = ruse_primitive_tasks[ruse_primitive_tasks["found"].notnull()]

In [ ]:
def get_real_sy_path(name):
    sobeq_benchmarks_dir = os.path.join(get_workspace_root(), "experiments/ruse_sobeq_benchmarks/benchmarks")
    return pathlib.Path(sobeq_benchmarks_dir).glob(f'**/{os.path.basename(name)}').__next__()

def parse_sobeq_results(path):
    sobeq_tasks = pd.read_csv(path)
    sobeq_tasks = sobeq_tasks[sobeq_tasks["store type"] == "Subsume"]
    sobeq_tasks.pop("store type")
    sobeq_tasks["path"] = sobeq_tasks["name"].apply(get_real_sy_path)
    sobeq_tasks["name"] = sobeq_tasks["path"].apply(lambda x: os.path.basename(x).split(".")[0])
    sobeq_tasks["expected"] = sobeq_tasks["path"].apply(lambda x: get_sy_info(x)["solution"])
    sobeq_tasks["num_examples"] = sobeq_tasks["path"].apply(lambda x: len(get_sy_info(x)["examples"]))
    sobeq_tasks["total_time"] = sobeq_tasks["time (ms)"].apply(lambda x: x / 1000)

    return sobeq_tasks

def parse_frangel_primitive_results(path):
    frangel_tasks = pd.read_csv(path)
    frangel_tasks["path"] = frangel_tasks["name"].apply(get_real_sy_path)
    frangel_tasks["name"] = frangel_tasks["path"].apply(lambda x: os.path.basename(x).split(".")[0])
    frangel_tasks["expected"] = frangel_tasks["path"].apply(lambda x: get_sy_info(x)["solution"])
    frangel_tasks["num_examples"] = frangel_tasks["path"].apply(lambda x: len(get_sy_info(x)["examples"]))
    frangel_tasks["total_time"] = frangel_tasks["time (ms)"].apply(lambda x: x / 1000)

    return frangel_tasks

def parse_frangel_oop_results(path):
    frangel_tasks = pd.read_csv(path)
    # TODO: fix this
    frangel_tasks["num_examples"] = 1
    frangel_tasks["total_time"] = frangel_tasks["time (ms)"].apply(lambda x: x / 1000)
    return frangel_tasks

def get_sobeq_solved_tasks(sobeq_tasks: pd.DataFrame):
    return sobeq_tasks[sobeq_tasks["solution"] != "Timeout"]

def get_frangel_solved_tasks(frangel_tasks: pd.DataFrame):
    frangel_solved_tasks_grouped = frangel_tasks[frangel_tasks["solution"] != "Timeout"].groupby("name")
    def se_minutes(x): return np.std(x/60)/np.sqrt(len(x))
    def se_seconds(x): return np.std(x)/np.sqrt(len(x))
    frangel_solved_tasks = frangel_solved_tasks_grouped.agg(
        num_examples=("num_examples", "first"),
        total_time = ("total_time", "mean"),
        total_time_min = ("total_time", "min"),
        total_time_max = ("total_time", "max"),
        total_time_se_minutes = ("total_time", se_minutes),
        total_time_se_seconds = ("total_time", se_seconds),
    )
    frangel_solved_tasks.reset_index(inplace=True)
    return frangel_solved_tasks


In [ ]:
sobeq_tasks = parse_sobeq_results("results/sobeq_results.csv")
frangel_tasks = parse_frangel_primitive_results("results/frangel_results.csv")
frangel_oop_tasks = parse_frangel_oop_results("results/frangel_full_oop_results.csv")

sobeq_solved_tasks = get_sobeq_solved_tasks(sobeq_tasks)
frangel_solved_tasks = get_frangel_solved_tasks(frangel_tasks)
frangel_oop_solved_tasks = get_frangel_solved_tasks(frangel_oop_tasks)

sobeq_solved_tasks = sobeq_solved_tasks.sort_values(by="total_time")
frangel_solved_tasks = frangel_solved_tasks.sort_values(by="total_time")
frangel_oop_solved_tasks = frangel_oop_solved_tasks.sort_values(by="total_time")
ruse_primitive_solved_tasks = ruse_primitive_solved_tasks.sort_values(by="total_time")
ruse_full_oop_solved_tasks = ruse_full_oop_solved_tasks.sort_values(by="total_time")

display_df(frangel_solved_tasks)


In [ ]:
print(f"Ruse solved {len(ruse_primitive_solved_tasks)}/{len(ruse_primitive_tasks)} tasks")
print(f"FrAngel solved {len(frangel_solved_tasks)}/{len(frangel_tasks.groupby('name'))} tasks")
print(f"SObEq solved {len(sobeq_solved_tasks)}/{len(sobeq_tasks)} tasks")
print()
print(f"Ruse OOP solved {len(ruse_full_oop_solved_tasks)}/{len(ruse_full_oop_tasks)} tasks")
print(f"FrAngel OOP solved {len(frangel_oop_solved_tasks)}/{len(frangel_oop_tasks.groupby('name'))} tasks")


In [ ]:
def plot_comparison(solved_tasks: dict[str, pd.DataFrame], colors: dict[str, str], total_tasks: int, y: ([int], int), x_groups: [(int, int, [int])], figsize=(10, 10), transform_time=None):
    fig, all_axis = plt.subplots(
        1, len(x_groups), figsize=figsize, sharey=True, facecolor='w')
    fig.subplots_adjust(wspace=0.05)
    if len(x_groups) == 1:
        all_axis = [all_axis]

    solved_ticks = []

    yticks, y_max = y

    def get_times(series):
        times = transform_time(series) if transform_time else series
        times = np.array(times)
        times.sort()

        return times

    for ax, (x_start, x_end, xticks) in zip(all_axis, x_groups):
        ax.set_ylim(0, y_max)
        ax.set_xlim(x_start, x_end)
        ax.set_xticks(xticks)

        ax.hlines(y=total_tasks, xmin=x_start, xmax=x_end,
                    color="black", linestyle="--", label="Total tasks")

        for i, (name, df) in enumerate(solved_tasks.items()):
            times = get_times(df["total_time"])
            
            ax.plot(times, np.arange(1, len(times) + 1),
                    color=colors[name], label=name, marker="x")

            if 'total_time_min' in df.columns and 'total_time_max' in df.columns:
                min_times = get_times(df["total_time_min"])
                max_times = get_times(df["total_time_max"])
                
                # Combine min_times and max_times into a single sorted array of unique x values
                combined_times = np.concatenate((min_times, max_times, [x_end]))
                combined_times.sort(kind='mergesort')
                # For each x in combined_times, find the corresponding y value from min_times or max_times
                # If x is in min_times, use its index+1 as y_min; else, use previous y_min
                # Similarly for max_times
                min_times_y = np.searchsorted(min_times, combined_times, side='right')
                max_times_y = np.searchsorted(max_times, combined_times, side='right')

                ax.fill_between(combined_times, max_times_y, min_times_y,
                                color=colors[name], alpha=0.3)
            
            ax.hlines(y=len(times), xmin=times.max(),
                      xmax=x_end, color=colors[name])

            solved_ticks += [len(times)]


    if len(x_groups) == 1:
        ax_l = all_axis[0]
        ax_r = all_axis[0].twinx()
    else:
        ax_l = all_axis[0]
        ax_r = all_axis[-1].twinx()
    
    ax_r.set_ylim(0, y_max)
    ax_r.spines['left'].set_visible(False)
    ax_l.yaxis.tick_left()
    ax_r.yaxis.tick_right()
    ax_l.tick_params(axis='y', labelleft=True)
    ax_r.tick_params(axis='y', labelright=True)
    ax_l.set_yticks(yticks)
    ax_r.set_yticks(np.unique(yticks + solved_ticks))

    if len(x_groups) > 1:
        all_axis[0].spines['right'].set_visible(False)
        for i in range(1, len(all_axis)):
            all_axis[i].spines['right'].set_visible(False)
            all_axis[i].spines['left'].set_visible(False)
            all_axis[i].yaxis.set_visible(False)

        all_axis[-1].spines['left'].set_visible(False)

        d = .015  # how big to make the diagonal lines in axes coordinates
        # arguments to pass plot, just so we don't keep repeating them
        kwargs = dict(
            transform=all_axis[0].transAxes, color='k', clip_on=False)
        all_axis[0].plot((1-d, 1+d), (-d, +d), **kwargs)
        all_axis[0].plot((1-d, 1+d), (1-d, 1+d), **kwargs)
        for i in range(1, len(all_axis)-1):
            kwargs.update(transform=all_axis[i].transAxes)
            all_axis[i].plot((1-d, 1+d), (-d, +d), **kwargs)
            all_axis[i].plot((1-d, 1+d), (1-d, 1+d), **kwargs)
            all_axis[i].plot((-d, +d), (1-d, 1+d), **kwargs)
            all_axis[i].plot((-d, +d), (-d, +d), **kwargs)

        kwargs.update(transform=all_axis[-1].transAxes)
        all_axis[-1].plot((-d, +d), (1-d, 1+d), **kwargs)
        all_axis[-1].plot((-d, +d), (-d, +d), **kwargs)

    all_axis[-1].legend(loc="lower right")

os.makedirs("results/figures", exist_ok=True)

In [ ]:
colors = {
    "frangel": "tab:orange",
    "sobeq": "tab:green",
    "ruse": "tab:blue",
    "frangel_min": ("tab:orange", 0.3),
    "frangel_max": ("tab:orange", 0.3)
}

primitive_solved_tasks = {"frangel": frangel_solved_tasks,
                          "sobeq": sobeq_solved_tasks, 
                          "ruse": ruse_primitive_solved_tasks}
total_primitive_tasks = len(sobeq_tasks)

plot_comparison(primitive_solved_tasks, colors, total_primitive_tasks, 
                transform_time=lambda x: x / 60,
                y=(np.arange(0, 70, 10).tolist() + [total_primitive_tasks], total_primitive_tasks+5),
                x_groups=[(0, 6, np.arange(6)), (6, 65, np.arange(10, 65, 10))],
                figsize=(16, 10))

plt.savefig("results/figures/primitive_solved_tasks.png")

In [ ]:
plot_comparison(primitive_solved_tasks, colors, total_primitive_tasks, 
                transform_time=lambda x: x[x <= 7],
                y=(np.arange(0, 70, 10).tolist() + [total_primitive_tasks], total_primitive_tasks+5),
                x_groups=[(0, 7, np.arange(8))],
                figsize=(16, 10))
plt.savefig("results/figures/primitive_solved_tasks_cutoff.png")

In [ ]:
oop_solved_tasks = {"frangel": frangel_oop_solved_tasks,
                    "ruse": ruse_full_oop_solved_tasks}
total_oop_tasks = len(ruse_full_oop_tasks)

plot_comparison(oop_solved_tasks, colors, total_oop_tasks,  
                y=((np.arange(0, 20, 5).tolist() + [total_oop_tasks]), 26),
                x_groups=[(0, 65, np.arange(0, 66, 5))],
                figsize=(16, 10))

plt.savefig("results/figures/oop_solved_tasks.png")

### RQ2 - Can <span style="font-variant:small-caps;">Ruse</span> handle aliasing and relations between variables?

In [ ]:
relations_tasks = ruse_tasks[ruse_tasks["path"].str.contains("relations")]
display_df(relations_tasks[["name", "expected"]])


In [ ]:
inc_min_ops = relations_tasks[relations_tasks["name"].isin([   "inc_tree_values_min_ops", "inc_tree_values_connected_min_ops"])]
inc = relations_tasks[relations_tasks["name"].isin(["inc_tree_values", "inc_tree_values_connected"])]
aliasing = relations_tasks[relations_tasks["name"].isin(["user_names", "user_names_aliasing"])]
connected = relations_tasks[relations_tasks["name"].isin(["user_names", "user_names_connected"])]

def plot_bar_chart(df):
    fig, ax = plt.subplots(2, 2, figsize=(16, 10))
    ax[0][0].bar(df["name"], df["total_time"], color="C0")
    ax[0][0].title.set_text("Time")
    ax[0][1].bar(df["name"], df["iterations"].apply(lambda x: len(x)), color="C1")
    ax[0][1].title.set_text("Iterations")
    ax[1][0].bar(df["name"], df["stats.BankSize"], color="C2")
    ax[1][0].title.set_text("Bank Size")
    ax[1][1].bar(df["name"], df["stats.FoundContextCount"], color="C3")
    ax[1][1].title.set_text("Found Context Count")
    ax[1][1].locator_params(axis='y', integer=True)

In [ ]:
plot_bar_chart(inc_min_ops)
plt.show()

In [ ]:
plot_bar_chart(inc)
plt.show()

In [ ]:
plot_bar_chart(aliasing)
plt.show()

In [ ]:
plot_bar_chart(connected)
plt.show()

### RQ3 - <span style="font-variant:small-caps;">Ruse</span> Embedding overhead

In [ ]:
overhead = pd.DataFrame(
    [("Without Embedding", 4642.17, datetime.timedelta(seconds=109)),
     ("With Embedding", 330.09, datetime.timedelta(seconds=57 * 60 + 8))],
    columns=["", "iterations (M)", "time"])

display_df(overhead)